In [6]:
import pandas as pd
import re

# ── 1. Chargement ──────────────────────────────────────────────────────────────
df = pd.read_csv(
    "scraping_pap.csv",
    on_bad_lines="skip",
    sep=None,
    engine="python"
)

# Renommer la colonne type (BOM possible en début de fichier)
df.columns = [c.strip().strip('\ufeff').strip('"') for c in df.columns]

# ── 2. Séparer "type" → "transaction" + "type_bien" ───────────────────────────
# Format source : "ventes-appartements", "locations-maisons", etc.
df[["transaction", "type_bien"]] = df["type"].str.split("-", n=1, expand=True)
df.drop(columns=["type"], inplace=True)

# ── 3. Ajouter la colonne source ───────────────────────────────────────────────
df["source"] = "Pap"

# ── 4. Extraire code postal et ville depuis la colonne "ville" ─────────────────
# Cas 1 : "Paris 5E (75005)"      → ville=Paris,       cp=75005
# Cas 2 : "Maison ... Saint-Cloud (92210)" → ville=Saint-Cloud, cp=92210
# Stratégie : le CP est toujours entre parenthèses ; la ville est le mot
# (pouvant contenir des tirets) juste avant la parenthèse ouvrante.

def extract_ville_cp(s):
    if pd.isna(s):
        return pd.NA, pd.NA
    s = s.replace('\xa0', ' ')  # espace insécable → espace normal

    # Code postal : toujours entre parenthèses
    cp_match = re.search(r'\((\d{5})\)', s)
    cp = cp_match.group(1) if cp_match else pd.NA

    # Mot juste avant "(" — peut être un arrondissement (ex: 5E, 10E)
    ville_match = re.search(r'([\w-]+)\s+([\w-]+)\s*\(\d{5}\)', s)
    if ville_match:
        avant_dernier = ville_match.group(1)  # ex: "Paris" ou "Saint-Cloud"
        dernier       = ville_match.group(2)  # ex: "5E" ou "(ignoré si ville composée)"
        # Si le dernier mot ressemble à un arrondissement (chiffre + lettre)
        if re.match(r'^\d+\w*$', dernier):
            ville = avant_dernier
        else:
            ville = dernier
    else:
        # Fallback : un seul mot avant la parenthèse
        m = re.search(r'([\w-]+)\s*\(\d{5}\)', s)
        ville = m.group(1) if m else pd.NA

    return ville, cp

df[["ville_clean", "code_postal"]] = df["ville"].apply(
    lambda s: pd.Series(extract_ville_cp(s))
)
df.drop(columns=["ville"], inplace=True)
df.rename(columns={"ville_clean": "ville"}, inplace=True)

# ── 5. Surface → float ────────────────────────────────────────────────────────
df["surface"] = df["surface"].astype(float)

# ── 6. nb_pieces, nb_chambres, prix → Int64 (nullable integer) ────────────────
# Int64 (majuscule) supporte les NaN contrairement à int64 (minuscule)
for col in ["nb_pieces", "nb_chambres", "prix"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

# ── 7. Vérification rapide ─────────────────────────────────────────────────────
print(df.dtypes)
print()
print(df[["transaction", "type_bien", "ville", "code_postal", "surface",
          "nb_pieces", "nb_chambres", "prix", "source"]].head(10))

prix             Int64
surface        float64
nb_pieces        Int64
nb_chambres      Int64
description     object
transaction     object
type_bien       object
source          object
ville           object
code_postal     object
dtype: object

  transaction     type_bien  ville code_postal  surface  nb_pieces  \
0      ventes  appartements  Paris       75005     31.0          2   
1      ventes  appartements  Paris       75002     64.0          2   
2      ventes  appartements  Paris       75005    107.4          4   
3      ventes  appartements  Paris       75010     60.0          3   
4      ventes  appartements  Paris       75007     30.0          2   
5      ventes  appartements  Paris       75015     11.0          1   
6      ventes  appartements  Paris       75003     70.0          4   
7      ventes  appartements  Paris       75017     63.0          3   
8      ventes  appartements  Paris       75010     19.0          1   
9      ventes  appartements  Paris       75013     72.4

In [13]:
df.head()

,prix,surface,nb_pieces,nb_chambres,description,transaction,type_bien,source,ville,code_postal
0,390000,31.0,2,1,"Très bien situé, tout commerces au pied de l'i...",ventes,appartements,Pap,Paris,75005
1,770000,64.0,2,1,**Très bel appartement traversant** situé au 5...,ventes,appartements,Pap,Paris,75002
2,1090000,107.4,4,3,**Grand duplex de charme quartier Gobelins Jar...,ventes,appartements,Pap,Paris,75005
3,630000,60.0,3,1,"Appartement 3 pièces 60 m2, proche Place Franz...",ventes,appartements,Pap,Paris,75010
4,425000,30.0,2,1,Appartement de 30 m² situé au rez-de-chaussée ...,ventes,appartements,Pap,Paris,75007


In [12]:
df.to_csv("C:/Users/julin/OneDrive/Documents/M2 TIDE/S2/big data/PAP_clean.csv",sep=";", index=False,encoding="utf-8-sig", quoting=1)